# 第15章 機械学習アプリの作り方

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍のリスト番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/appendix/

## モデルの保存と再利用

### pickleによるモデルの保存と読み込み

**リスト 15.1**　`pickle`によるモデルの保存

In [ ]:
import pickle
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# モデルの学習（iris は seaborn 版で読み込み）
df_iris = sns.load_dataset("iris")
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X_train, X_test, y_train, y_test = train_test_split(
    df_iris[feature_cols].values, df_iris["species"],
    test_size=0.2, random_state=42
)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f"学習時の精度: {model.score(X_test, y_test):.4f}")

# モデルをファイルに保存
with open("iris_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("モデルを iris_model.pkl に保存しました")

**リスト 15.2**　保存したモデルの読み込みと予測

In [ ]:
import pickle
import numpy as np

# モデルの読み込み
with open("iris_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# 読み込んだモデルで予測
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
prediction = loaded_model.predict(sample)
proba = loaded_model.predict_proba(sample)

print(f"予測結果: {prediction[0]}")
print(f"確率: {dict(zip(loaded_model.classes_, proba[0].round(4)))}")

### joblibによる保存

**リスト 15.3**　`joblib`によるモデルの保存と読み込み

In [ ]:
import joblib

# モデルの保存
joblib.dump(model, "iris_model.joblib")
print("モデルを iris_model.joblib に保存しました")

# モデルの読み込み
loaded_model = joblib.load("iris_model.joblib")
print(f"読み込んだモデルの精度: {loaded_model.score(X_test, y_test):.4f}")

## 推論処理の作り方

### 予測用モデルの準備

**リスト 15.4**　ペンギン分類モデルの学習と保存

In [ ]:
# train_penguin.py
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# データの準備（推論時は NumPy 配列を渡すため、.values で学習しておく）
penguins = sns.load_dataset("penguins").dropna()
X = penguins[["bill_length_mm", "bill_depth_mm",
              "flipper_length_mm", "body_mass_g"]].values
y = penguins["species"]

# パイプラインの構築と学習
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=100, random_state=42
    ))
])
pipeline.fit(X, y)

# モデルの保存
joblib.dump(pipeline, "penguin_model.joblib")
print("モデルを保存しました")
print(f"クラス: {list(pipeline.classes_)}")

### 推論関数の作成

**リスト 15.5**　推論関数`predict_penguin`の作成

In [ ]:
# predict.py
import joblib
import numpy as np

# モデルの読み込み（アプリ起動時に1回だけ実行する想定）
model = joblib.load("penguin_model.joblib")

# 特徴量からペンギンの種類と各クラスの確率を返す
def predict_penguin(bill_length, bill_depth,
                    flipper_length, body_mass):
    features = np.array([[bill_length, bill_depth,
                          flipper_length, body_mass]])
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    proba_dict = {
        species: round(float(prob), 4)
        for species, prob in zip(model.classes_, probabilities)
    }
    return prediction, proba_dict

# 動作確認
species, proba = predict_penguin(39.1, 18.7, 181.0, 3750.0)
print(f"予測結果: {species}")
print(f"確率: {proba}")